In [1]:
# Check data

import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/GSE189149_counts_table.txt', sep='\t', index_col=0)

print('Min value:', df.values.min())
print('Max value:', df.values.max())
print('Mean value:', df.values.mean().round(3))
print('First sample sum:', df.iloc[:,0].sum().round(1))

Min value: 0.0
Max value: 11968.1816
Mean value: 49.395
First sample sum: 1000000.0


Normalizing the Gene Counts

In [2]:
# Log2 transform the data to normalize for EVA
df_log2 = np.log2(df+1)

# Check values after log2-ing them
print('Min value after log2:', df_log2.values.min().round(2))
print('Max value after log2:', df_log2.values.max().round(2))
print('Mean value after log2:', df_log2.values.mean().round(2))

Min value after log2: 0.0
Max value after log2: 13.55
Mean value after log2: 2.99


In [3]:
# Check current gene naming

print("First 10 gene names:", df_log2.index[:10].tolist())
print("All genes:", len(df_log2.index))

First 10 gene names: ['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112', 'FGR', 'CFH', 'FUCA2', 'GCLC', 'NFYA']
All genes: 20245


In [4]:
# Pivoting...I'm going to copy exactly the way the Colab tutorial for EVA does it
# Convert to numpy array
counts = df.values.T  # samples as rows, genes as columns

# Normalize to target sum then log2 transform
TARGET_SUM = 1e6
counts = counts / counts.sum(axis=1, keepdims=True) * TARGET_SUM
counts = np.log1p(counts) / np.log(2)

print("Shape:", counts.shape)
print("Max value:", counts.max().round(3))
print("Mean value:", counts.mean().round(3))

Shape: (48, 20245)
Max value: 13.547
Mean value: 2.99


In [5]:
# Just realized I haven't worked out the resting v activated issue, so let's see the whole thing
for i, col in enumerate(df.columns):
    print(i, col)

# Okay, looks like we're good

0 X2202_M_NA
1 X2207_F_NA
2 X3752_M_NA
3 X3762_M_NA
4 X3874_M_NA
5 X4219_F_NA
6 X4502_M_NA
7 X4730_F_NA
8 X5603_M_NA
9 X5827_M_NA
10 X6341_F_NA
11 X6398_M_NA
12 X7034_M_NA
13 X7076_F_NA
14 X7346_F_NA
15 X7414_M_NA
16 X7524_M_NA
17 X8977_F_NA
18 X10540_F_PA
19 X1728_M_PA
20 X1847_M_PA
21 X2065_F_PA
22 X4366_M_PA
23 X4872_M_PA
24 X5868_M_PA
25 X6081_M_PA
26 X7019_M_PA
27 X7022_M_PA
28 X7087_F_PA
29 X8798_F_PA
30 X9470_M_PA
31 X1337_M_MA
32 X1535_M_MA
33 X1794_M_MA
34 X2131_F_MA
35 X2138_F_MA
36 X2627_M_MA
37 X2712_M_MA
38 X4189_F_MA
39 X4389_M_MA
40 X4413_M_MA
41 X4759_F_MA
42 X566_F_MA
43 X5757_F_MA
44 X7426_F_MA
45 X7525_F_MA
46 X8072_M_MA
47 X9620_F_MA


Mapping GeneIDs

In [6]:
# Load EVA tokenizer 
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("ScientaLab/eva-rna", trust_remote_code=True)

# Grab symbol to NCBI mapper
mapper = tokenizer.gene_mapper["symbol_to_ncbi"]

# Get the list of gene symbols from df
gene_symbols = df.index.tolist()

# Check how many genes actually have a match in mapper
found = [g for g in gene_symbols if g in mapper]
missing = [g for g in gene_symbols if g not in mapper]

print(f"Total genes: {len(gene_symbols)}")
print(f"Found in EVA vocabulary: {len(found)}")
print(f"Missing: {len(missing)}")

/Users/zaminsmac/miniconda3/envs/eva-allergy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total genes: 20245
Found in EVA vocabulary: 19088
Missing: 1157


In [7]:
# Forgot to actually install transformers, so we're running it back
# Load EVA tokenizer 
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("ScientaLab/eva-rna", trust_remote_code=True)

# Grab symbol to NCBI mapper
mapper = tokenizer.gene_mapper["symbol_to_ncbi"]

# Get the list of gene symbols from df
gene_symbols = df.index.tolist()

# Check how many of your genes actually have a match in EVA's vocabulary
found = [g for g in gene_symbols if g in mapper]
missing = [g for g in gene_symbols if g not in mapper]

print(f"Total genes: {len(gene_symbols)}")
print(f"Found in EVA vocabulary: {len(found)}")
print(f"Missing: {len(missing)}")

# Check the names of some of the missing genes
print(missing[:10])

Total genes: 20245
Found in EVA vocabulary: 19088
Missing: 1157
['AC004381.6', 'MVP', 'GUCA1A', 'VDAC3', 'Mar-02', 'Sep-03', 'XBP1', 'MIEF1', 'TEX28P2', 'FMR1']


Initial Manual Approach to normalize gene names before running through EVA. However, manual approach would not sufficiently resolve all 1157 gene names (attempted hyphen + date approach), so later decided to use mygene.

In [8]:
# Check XBP1 gene because it's important and should have a correlation in mapper
print("XBP1" in mapper)
print("xbp1" in mapper)
print("XBP-1" in mapper)

False
False
True


In [9]:
# Figured out the issue, some are configured differently. After some research, I realized there are multiple different naming conventions, and this one may be outdated, so let's see what keys are actually in the EVA tokenizer
print(tokenizer.gene_mapper.keys())

dict_keys(['ensembl_to_ncbi', 'symbol_to_ncbi', 'ensembl_to_ncbi_mouse', 'symbol_to_ncbi_mouse'])


In [10]:
# Try to write a function to try all possible configurations, starting with digit at end needs hyphen
def try_hyphen_variant(symbol):
    if symbol[-1].isdigit():
        new_string = symbol[:-1] + "-" + symbol[-1]
    else: 
        new_string = symbol
    if new_string in mapper:
        return (new_string)
    pass

print(try_hyphen_variant('XBP1'))   
print(try_hyphen_variant('MVP'))

XBP-1
None


In [11]:
# Now fix date issue
month_map = {
    'Mar': 'MARCH', 'Sep': 'SEPT', 'Sept': 'SEPT', 'Dec': 'DEC', 'Oct': 'OCT',
}

def fix_date_corrupted_gene(symbol):
    if '-' in symbol:
        parts = symbol.split('-')
        if len(parts) == 2:
            month_part, day_part = parts
            if month_part in month_map and day_part.isdigit():
                day_num = int(day_part)
                return f"{month_map[month_part]}{day_num}"
    return symbol

print(fix_date_corrupted_gene("Mar-01"))

MARCH1


In [12]:
rescued = {}      # Keep map for later
still_missing = []

for gene in missing:
    # Try date-fix function 
    candidate = fix_date_corrupted_gene(gene)
    
    # Check if it's now in the mapper
    if candidate in mapper:
        rescued[gene] = candidate
        continue   # skip to the next gene, this one's done
    
    # Otherwise fix the hyphen issue
    hyphen_try = try_hyphen_variant(candidate)
    if hyphen_try is not None:
        rescued[gene] = hyphen_try
        continue
    
    # Add it back to still missing if it didn't work
    still_missing.append(gene)

print(f"Rescued: {len(rescued)}")
print(f"Still missing: {len(still_missing)}")

Rescued: 28
Still missing: 1127


Using mygene to get to the NCBI number directly instead of manually going through all the missing genes

In [13]:
import mygene

mg = mygene.MyGeneInfo()

test_genes = ['XBP1', 'MVP', 'FMR1', 'VDAC3', 'MIEF1']

results = mg.querymany(test_genes, scopes='symbol,alias', species='human', fields='symbol,entrezgene,_score')

for r in results:
    print(r)

4 input query terms found dup hits:	[('XBP1', 2), ('MVP', 2), ('FMR1', 3), ('VDAC3', 2)]


{'query': 'XBP1', '_id': '7494', '_score': 16.920956, 'entrezgene': '7494', 'symbol': 'XBP1'}
{'query': 'XBP1', '_id': '7495', '_score': 6.889792, 'entrezgene': '7495', 'symbol': 'XBP1P1'}
{'query': 'MVP', '_id': '50951', '_score': 21.442438, 'entrezgene': '50951', 'symbol': 'MMVP1'}
{'query': 'MVP', '_id': '9961', '_score': 17.82573, 'entrezgene': '9961', 'symbol': 'MVP'}
{'query': 'FMR1', '_id': '106478973', '_score': 21.625181, 'entrezgene': '106478973', 'symbol': 'FMR1-IT1'}
{'query': 'FMR1', '_id': '108684022', '_score': 21.625181, 'entrezgene': '108684022', 'symbol': 'FRAXA'}
{'query': 'FMR1', '_id': '2332', '_score': 16.98725, 'entrezgene': '2332', 'symbol': 'FMR1'}
{'query': 'VDAC3', '_id': '7419', '_score': 17.538168, 'entrezgene': '7419', 'symbol': 'VDAC3'}
{'query': 'VDAC3', '_id': '10187', '_score': 7.260311, 'entrezgene': '10187', 'symbol': 'VDAC1P5'}
{'query': 'MIEF1', '_id': '54471', '_score': 17.594387, 'entrezgene': '54471', 'symbol': 'MIEF1'}


In [14]:
# Check results
print(mapper['XBP-1'])

7494


In [15]:
# Build function to return entrezgene if symbol matches query in mygene
def pick_best_hit(query_gene, all_results):
    found = [r for r in all_results if r['query'] == query_gene]
    for hit in found:
        if 'symbol' in hit and hit['symbol'] == query_gene and 'entrezgene' in hit:
            return hit['entrezgene']
    return None

print(pick_best_hit('FMR1', results))
print(pick_best_hit('MVP', results))


2332
9961


In [16]:
# Check whether they're even in results
for r in results:
    if 'symbol' not in r:
        print(r)
        break

for r in results:
    if 'symbol' in r and 'entrezgene' not in r:
        print(r)
        break

In [17]:
# Now, run everything through the new function
results = mg.querymany(still_missing, scopes='symbol,alias', species='human', fields='symbol,entrezgene,_score')

mygene_rescued = {}
mygene_still_missing = []

for gene in still_missing:
    hit_id = pick_best_hit(gene, results)
    if hit_id is not None:
        mygene_rescued[gene] = hit_id
    else:
        mygene_still_missing.append(gene)

print(f"Rescued by mygene: {len(mygene_rescued)}")
print(f"Still missing: {len(mygene_still_missing)}")


39 input query terms found dup hits:	[('MVP', 2), ('FMR1', 3), ('ELP4', 2), ('MDK', 2), ('RGS2', 2), ('FGD3', 2), ('APOBEC3A', 2), ('HTR2
987 input query terms found no hit:	['AC004381.6', 'DKFZP761J1410', 'AL021546.6', 'RP11-268J15.5', 'RP11-408E5.4', 'KIAA1045', 'RP11-51F


Rescued by mygene: 115
Still missing: 1012


In [18]:
# Okay, now let's generate a list of all the genes that have worked so far

final_gene_ids = {}
truly_unresolved = []

for gene in df.index:
    if gene in mapper:
        final_gene_ids[gene] = mapper[gene]
    elif gene in rescued:
        corrected_symbol = rescued[gene]
        final_gene_ids[gene] = mapper[corrected_symbol]   
    elif gene in mygene_rescued:
        final_gene_ids[gene] = mygene_rescued[gene]
    else:
        truly_unresolved.append(gene)

print(f"Finally resolved: {len(final_gene_ids)}")
print(f"Unfortunately left behind: {len(truly_unresolved)}")



Finally resolved: 19231
Unfortunately left behind: 1012


In [19]:
# Filtering out the genes that didn't match
keep_positions = []   
ordered_gene_ids = [] 

for position, gene in enumerate(df.index):
    if gene in final_gene_ids:   
        keep_positions.append(position)
        ordered_gene_ids.append(final_gene_ids[gene])  

eva_input_matrix = counts[:, keep_positions]  


print(eva_input_matrix.shape)      
print(len(ordered_gene_ids))  

print(ordered_gene_ids[:10])

(48, 19233)
19233
['7105', '64102', '8813', '57147', '55732', '2268', '3075', '2519', '2729', '4800']


In [20]:
# Figure out why there are two missing...probably duplicates
print(df.index.duplicated().sum())
print(df.index[df.index.duplicated(keep=False)])

2
Index(['Mar-02', 'Mar-02', 'Mar-01', 'Mar-01'], dtype='object')


In [21]:
# Okay, we're going to have to just boot these four for now since there's no way to tell from the file which is MARCH and which is MTARC
excluded_genes = {'Mar-01', 'Mar-02'}  # known unresolvable Excel date-corruption collisions

keep_positions = []
ordered_gene_ids = []

for position, gene in enumerate(df.index):
    if gene in final_gene_ids and gene not in excluded_genes:
        keep_positions.append(position)
        ordered_gene_ids.append(final_gene_ids[gene])

eva_input_matrix = counts[:, keep_positions]
print(eva_input_matrix.shape)
print(len(ordered_gene_ids))

(48, 19229)
19229


Preparing Data for Model (SelectK)

In [22]:
labels = []

for col in df.columns:
    suffix = col.split('_')[-1]   
    
    if suffix == 'NA':
        labels.append('healthy_control')   
    elif suffix == 'PA' or suffix == 'MA':       
        labels.append('allergic')   

print(len(labels))          # should be 48
print(labels[:5])          
print(labels.count('healthy_control')) # should be 18

48
['healthy_control', 'healthy_control', 'healthy_control', 'healthy_control', 'healthy_control']
18


In [23]:
from sklearn.feature_selection import SelectKBest, f_classif

# SelectKBest needs numeric labels, not text strings
numeric_labels = [1 if label == 'allergic' else 0 for label in labels]

selector = SelectKBest(score_func=f_classif, k=2000)
reduced_matrix = selector.fit_transform(eva_input_matrix, numeric_labels)

# Get which gene positions were kept, so we can filter ordered_gene_ids to match
selected_mask = selector.get_support()   
reduced_gene_ids = [gene_id for gene_id, keep in zip(ordered_gene_ids, selected_mask) if keep]

print(reduced_matrix.shape)       # should be (48, 2000)
print(len(reduced_gene_ids))      # should be 2000

(48, 2000)
2000


/Users/zaminsmac/miniconda3/envs/eva-allergy/lib/python3.10/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [    1    22    68 ... 19216 19223 19226] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/zaminsmac/miniconda3/envs/eva-allergy/lib/python3.10/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [24]:
# Save everything
import pickle
with open('gene_selection_checkpoint_FIXED.pkl', 'wb') as f:
    pickle.dump({'reduced_matrix': reduced_matrix, 'reduced_gene_ids': reduced_gene_ids, 'labels': labels, 'numeric_labels': numeric_labels}, f)

In [25]:
#Check how many were constant
import numpy as np
constant_count = np.sum(np.all(eva_input_matrix == eva_input_matrix[0, :], axis=0))
print(f"Constant (zero-variance) genes: {constant_count}")

Constant (zero-variance) genes: 2257


Load Model, then Generating Embeddings

In [26]:
from transformers import AutoModel

model = AutoModel.from_pretrained("ScientaLab/eva-rna", trust_remote_code=True)

Loading weights: 100%|█████████████████████| 319/319 [00:00<00:00, 9778.72it/s]


In [ ]:
import torch

batch_gene_ids = [reduced_gene_ids for _ in range(reduced_matrix.shape[0])]
batch_expression = reduced_matrix.tolist()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()

inputs = tokenizer(batch_gene_ids, batch_expression, padding=True, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.inference_mode():
    outputs = model(**inputs)

eva_features = outputs.cls_embedding.cpu().numpy()
print(eva_features.shape)

/Users/zaminsmac/.cache/huggingface/modules/transformers_modules/ScientaLab/eva_hyphen_rna/24077343116ad86c59de607c1e4e04299893beb1/tokenization_eva_rna.py:189: UserWarning: Token '642489' not in vocabulary, falling back to pad_token_id.
  token_ids = [self._convert_token_to_id(str(g)) for g in genes]


In [ ]:
with open('eva_features_FINAL.pkl', 'wb') as f:
    pickle.dump(eva_features, f)

Building and Evaluating Classifier on Model Embeddings

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_lr = LogisticRegression(max_iter=1000)

eva_auc_scores = cross_val_score(
    model_lr, 
    eva_features,           
    numeric_labels,           
    cv=cv, 
    scoring='roc_auc'
)

print("Fold AUCs:", eva_auc_scores)
print("Mean AUC:", eva_auc_scores.mean().round(3))
print("Std Dev:", eva_auc_scores.std().round(3))

Checking rigor

In [ ]:
from sklearn.model_selection import permutation_test_score


# Let's check the AUC result
true_score, permutation_scores, p_value = permutation_test_score(
    model_lr,             
    eva_features,              
    numeric_labels,             
    cv=cv,
    scoring='roc_auc',
    n_permutations=1000,
    random_state=42,
)

print("True AUC:", true_score.round(3))
print("P-value:", p_value)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

pipeline = Pipeline([
    ('pca', PCA(n_components=20)),   
    ('logreg', LogisticRegression(max_iter=1000)),
])

eva_pca_auc_scores = cross_val_score(
    pipeline,        
    eva_features,
    numeric_labels,
    cv=cv,
    scoring='roc_auc',
)

print("Fold AUCs:", eva_pca_auc_scores)
print("Mean AUC:", eva_pca_auc_scores.mean().round(3))
print("Std Dev:", eva_pca_auc_scores.std().round(3))

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
eva_features_scaled = scaler.fit_transform(eva_features)

# idk what to put here, but felt like it needed a label
pca = PCA(n_components=20)
eva_features_pcan = pca.fit_transform(eva_features_scaled)
print("Explained variance ratio sum:", pca.explained_variance_ratio_.sum())

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the range of C values to search over
param_grid = {'logreg__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]}

# The inner search: for a GIVEN training set, find the best C using its own cross-validation
inner_search = GridSearchCV(
    pipeline,            
    param_grid,
    cv= 5,            
    scoring='roc_auc',
)


nested_auc_scores = cross_val_score(
    inner_search,               
    eva_features,
    numeric_labels,
    cv= 5,            
    scoring='roc_auc',
)

print("Nested Fold AUCs:", nested_auc_scores)
print("Nested Mean AUC:", nested_auc_scores.mean().round(3))

In [ ]:
inner_search.fit(eva_features, numeric_labels)
print("Best C found:", inner_search.best_params_)

In [ ]:
results_df = pd.DataFrame(inner_search.cv_results_)
print(results_df[['param_logreg__C', 'mean_test_score', 'std_test_score']])

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold

pipeline_final = Pipeline([
    ('pca', PCA(n_components=20)),
    ('logreg', LogisticRegression(max_iter=1000, C=100)),
])

repeated_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

final_auc_scores = cross_val_score(pipeline_final, eva_features, numeric_labels, cv=repeated_cv, scoring='roc_auc')

print("Mean AUC:", final_auc_scores.mean().round(3))
print("Std Dev:", final_auc_scores.std().round(3))

In [ ]:
pipeline_baseline_1 = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000)),
])

baseline_1_auc_scores = cross_val_score(
    pipeline_baseline_1,
    reduced_matrix,      # <-- your SelectKBest output, not EVA's
    numeric_labels,
    cv=cv,
    scoring='roc_auc',
)

print("Fold AUCs:", baseline_1_auc_scores)
print("Mean AUC:", baseline_1_auc_scores.mean().round(3))
print("Std Dev:", baseline_1_auc_scores.std().round(3))

In [ ]:
true_score_1, _, p_value_1 = permutation_test_score(
    pipeline_baseline_1, reduced_matrix, numeric_labels,
    cv=cv, scoring='roc_auc', n_permutations=1000, random_state=42,
)
print("Baseline (no PCA) — True AUC:", true_score_1.round(3), "P-value:", p_value_1)

In [ ]:
pipeline_baseline_2 = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=20)),
    ('logreg', LogisticRegression(max_iter=1000)),
])

baseline_2_auc_scores = cross_val_score(
    pipeline_baseline_2,
    reduced_matrix,      
    numeric_labels,
    cv=cv,
    scoring='roc_auc',
)

print("Fold AUCs:", baseline_2_auc_scores)
print("Mean AUC:", baseline_2_auc_scores.mean().round(3))
print("Std Dev:", baseline_2_auc_scores.std().round(3))

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
eva_features_scaled = scaler.fit_transform(reduced_matrix)

# idk what to put here, but felt like it needed a label
pca = PCA(n_components=20)
eva_features_pcan = pca.fit_transform(reduced_matrix)
print("Explained variance ratio sum:", pca.explained_variance_ratio_.sum())

In [ ]:
true_score_2, _, p_value_2 = permutation_test_score(
    pipeline_baseline_2, reduced_matrix, numeric_labels,   # which pipeline variable name did you use?
    cv=cv, scoring='roc_auc', n_permutations=1000, random_state=42,
)
print("Baseline (with PCA) — True AUC:", true_score_2.round(3), "P-value:", p_value_2)

In [ ]:
# Pipeline 1: SelectKBest + Scaler + LogReg (no PCA)
pipeline_baseline = Pipeline([
    ('selectk', SelectKBest(score_func=f_classif, k=2000)),
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000)),
])

import warnings
warnings.filterwarnings('ignore', category=UserWarning)

baseline_auc_scores = cross_val_score(
    pipeline_baseline, eva_input_matrix, numeric_labels,
    cv=cv, scoring='roc_auc',
)
print("Baseline (no PCA) Fold AUCs:", baseline_auc_scores)
print("Baseline (no PCA) Mean AUC:", baseline_auc_scores.mean().round(3))

# Pipeline 2: SelectKBest + Scaler + PCA + LogReg
pipeline_baseline_pca = Pipeline([
    ('selectk', SelectKBest(score_func=f_classif, k=2000)),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=20)),
    ('logreg', LogisticRegression(max_iter=1000)),
])

baseline_pca_auc_scores = cross_val_score(
    pipeline_baseline_pca, eva_input_matrix, numeric_labels,
    cv=cv, scoring='roc_auc',
)
print("Baseline (with PCA) Fold AUCs:", baseline_pca_auc_scores)
print("Baseline (with PCA) Mean AUC:", baseline_pca_auc_scores.mean().round(3))

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import torch
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def get_eva_embeddings(gene_ids_list, expression_matrix):
    batch_gene_ids = [gene_ids_list for _ in range(expression_matrix.shape[0])]
    batch_expression = expression_matrix.tolist()
    inputs = tokenizer(batch_gene_ids, batch_expression, padding=True, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.inference_mode():
        outputs = model(**inputs)
    return outputs.cls_embedding.cpu().numpy()

y_arr = np.array(numeric_labels)
eva_nested_auc = []

for train_idx, test_idx in cv.split(eva_input_matrix, y_arr):
    X_train, X_test = eva_input_matrix[train_idx], eva_input_matrix[test_idx]
    y_train, y_test = y_arr[train_idx], y_arr[test_idx]

    selector_fold = SelectKBest(score_func=f_classif, k=2000)
    selector_fold.fit(X_train, y_train)
    mask = selector_fold.get_support()
    fold_gene_ids = [gid for gid, keep in zip(ordered_gene_ids, mask) if keep]

    X_train_sel = X_train[:, mask]
    X_test_sel = X_test[:, mask]

    train_embeddings = get_eva_embeddings(fold_gene_ids, X_train_sel)
    test_embeddings = get_eva_embeddings(fold_gene_ids, X_test_sel)

    clf = LogisticRegression(max_iter=1000)
    clf.fit(train_embeddings, y_train)
    probs = clf.predict_proba(test_embeddings)[:, 1]
    auc = roc_auc_score(y_test, probs)
    eva_nested_auc.append(auc)

eva_nested_auc = np.array(eva_nested_auc)
print("EVA (properly nested) Fold AUCs:", eva_nested_auc)
print("EVA (properly nested) Mean AUC:", eva_nested_auc.mean().round(3))

Maintenance

In [ ]:
import sys, os
print(sys.executable)
print(os.getcwd())

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/GSE189149_counts_table.txt', sep='\t', index_col=0)
print('Min value:', df.values.min())
print('Max value:', df.values.max())
print('Mean value:', df.values.mean().round(3))
print('First sample sum:', df.iloc[:,0].sum().round(1))

In [ ]:
print('Min value (counts):', counts.values.min().round(3))
print('Max value (counts):', counts.values.max().round(3))
print('Mean value (counts):', counts.values.mean().round(3))

In [ ]:
from huggingface_hub import login
login()